In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from crewai import LLM

llm = LLM(
  model = 'gemini/gemini-2.5-flash',
  temperature=0.1
)
llm.call("what is crew ai tell me in one line")

'Crew AI is a framework for building and orchestrating teams of specialized AI agents to collaboratively complete complex tasks.'

In [6]:
original_email ="""
Looping in Priya. TAS and PRX updates are in the deck. ETA for SDS integration is Friday.
Let's sync up tomorrow if SYNCBOT allows ping me if any blockers."""

from crewai.tools import BaseTool

class ReplaceJargonTool(BaseTool):
  name: str = "Replace Jargon"
  description: str = "Replaces jargon in the email with simpler language."
  
  def _run(self, email:str) -> str:
    replacements = {
      "PRX": "Project Phoenix (internal AI revamp project)",
      "TAS": "technical architecture stack",
      "DBX": "client database cluster",
      "SDS": "Smart Data Syncer",
      "SYNCBOT": "internal standup assistant bot",
      "WIP": "in progress",
      "POC": "proof of concept",
      "ping": "reach out"
    }
    suggestions = []
    email_lower = email.lower()
    for jargon, replacement in replacements.items():
      if jargon.lower() in email_lower:
        suggestions.append(f"Replace '{jargon}' with '{replacement}'")
    return "\n".join(suggestions) if suggestions else "No jargon found to replace."
  
jt = ReplaceJargonTool()
jt.run(original_email)


"Replace 'PRX' with 'Project Phoenix (internal AI revamp project)'\nReplace 'TAS' with 'technical architecture stack'\nReplace 'SDS' with 'Smart Data Syncer'\nReplace 'SYNCBOT' with 'internal standup assistant bot'\nReplace 'ping' with 'reach out'"

In [ ]:
from crewai import Agent, Task, Crew
# first we specifiy the agent
email_assitant = Agent(
  role= "email_assistant",
  goal= "Improve emails and make them sound professional and clear",
  backstory="A highly experience communication expoer skilled in professinal email writing.",
  verbose = True,
  llm = llm
)

# then we create a task for the agent to complete
email_task= Task(
  description=f"""Take the following rough email and improve it to make it sound more professional and clear: {original_email}""",
  agent=email_assitant,
  tools=[jt],
  expected_output='A professionally written email that clearly communicates the status of the demo and next steps.'
)

# then we create a crew to complete the task
crew = Crew(
  agents=[email_assitant],
  tasks=[email_task],
  verbose = True
)
result = crew.kickoff()
print(result)